In [2]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# 1) Load the data
data = pd.read_csv('../../../data/data_reduced_175_standard.csv')

print(data.columns)

teff = data['TEFF'].astype('float32')

# Drop les colonnes inutiles (apogee_id, etc.) SAUF TEFF
data = data.drop(columns=[
    'apogee_id', 'class_spectral', 'class_lum_logg', 'class_lum_jhk',
    'class_lum_bins_logg', 'class_lum_bins_jhk', 'M_H', 'VMICRO', 'VMACRO',
    'TEFF'  # On drop TEFF ici pour qu'elle ne fasse pas partie des features
])

# Convertir le reste en float32
data = data.astype('float32').reset_index(drop=True)

# data.shape doit maintenant être (N, nb_features_sans_TEFF)
print("Final Data Shape :", data.shape)


Index(['apogee_id', 'M_H', 'VMICRO', 'VMACRO', 'C_FE', 'CI_FE', 'N_FE', 'O_FE',
       'NA_FE', 'MG_FE', 'AL_FE', 'SI_FE', 'S_FE', 'K_FE', 'CA_FE', 'TI_FE',
       'V_FE', 'CR_FE', 'MN_FE', 'NI_FE', 'FE_H', 'class_spectral',
       'class_lum_logg', 'class_lum_jhk', 'class_lum_bins_logg',
       'class_lum_bins_jhk', 'TEFF'],
      dtype='object')
Final Data Shape : (175809, 17)


In [3]:
class AutoencoderRegressorDataset(Dataset):
    def __init__(self, features_df, teff_series):
        """
        features_df : DataFrame ou ndarray des features (dimensions [N, d])
        teff_series : Series ou ndarray de taille N
        """
        self.X = torch.tensor(features_df.values, dtype=torch.float32)
        self.teff = torch.tensor(teff_series.values, dtype=torch.float32)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        # x for l'encodeur/décodeur
        x_input = self.X[idx]          # dimension d
        # x_target for reconstruction
        x_target = self.X[idx]         # dimension d
        # teff for regression
        teff_value = self.teff[idx]    # scalaire
        return x_input, x_target, teff_value


In [4]:
# 1) Construire le dataset complet
full_dataset = AutoencoderRegressorDataset(data, teff)

# 2) Split en train / val
num_samples = len(full_dataset)
train_size = int(0.9 * num_samples)
val_size = num_samples - train_size

train_dataset, val_dataset = torch.utils.data.random_split(
    full_dataset,
    [train_size, val_size],
    generator=torch.Generator().manual_seed(42)
)

# 3) DataLoaders
train_loader = DataLoader(train_dataset, batch_size=1024, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=1024, shuffle=False)


In [5]:
class MultiTaskAutoencoder(nn.Module):
    def __init__(self, input_dim=17, latent_dim=2, hidden_reg=32):
        super(MultiTaskAutoencoder, self).__init__()

        # --- Encodeur ---
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(negative_slope=0.01),

            nn.Linear(32, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(negative_slope=0.01),

            nn.Linear(64, latent_dim)  # Pas d'activation ici
        )

        # --- Décodeur ---
        self.decoder = nn.Sequential(
            nn.Linear(latent_dim, 64),
            nn.BatchNorm1d(64),
            nn.LeakyReLU(negative_slope=0.01),

            nn.Linear(64, 32),
            nn.BatchNorm1d(32),
            nn.LeakyReLU(negative_slope=0.01),

            nn.Linear(32, input_dim)  # Reconstruction de dimension input_dim
        )

        # --- Branche de régression TEFF ---
        self.regressor = nn.Sequential(
            nn.Linear(latent_dim, hidden_reg),
            nn.LeakyReLU(negative_slope=0.01),
            nn.Linear(hidden_reg, 1)  # Sortie scalaire pour TEFF
        )

    def forward(self, x):
        # Encode
        z = self.encoder(x)
        # Décode
        x_recon = self.decoder(z)
        # Prédit TEFF
        teff_pred = self.regressor(z)
        return x_recon, teff_pred


In [1]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device :", device)

input_dim = data.shape[1]  # le nombre de features (après drop TEFF)
model = MultiTaskAutoencoder(input_dim=input_dim, latent_dim=2).to(device)

# Deux MSELoss possibles : un pour la reconstruction, un pour la régression
# (on peut utiliser la même fonction MSELoss si on veut)
mse_recon = nn.MSELoss()
mse_reg   = nn.MSELoss()

optimizer = optim.Adam(model.parameters(), lr=1e-3)

num_epochs = 100
patience = 5
best_val_loss = float('inf')
epochs_no_improve = 0

for epoch in range(num_epochs):
    model.train()
    train_loss = 0.0

    for X_batch, X_target, teff_batch in train_loader:
        X_batch = X_batch.to(device)
        X_target = X_target.to(device)
        teff_batch = teff_batch.to(device)

        optimizer.zero_grad()

        # Forward
        x_recon, teff_pred = model(X_batch)

        # Losses
        recon_loss = mse_recon(x_recon, X_target)
        reg_loss   = mse_reg(teff_pred.squeeze(), teff_batch)

        # Combiner
        loss = recon_loss + reg_loss  # alpha=1, beta=1

        # Backprop
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)

    train_loss /= len(train_loader.dataset)

    # --- Validation ---
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for X_batch, X_target, teff_batch in val_loader:
            X_batch = X_batch.to(device)
            X_target = X_target.to(device)
            teff_batch = teff_batch.to(device)

            x_recon, teff_pred = model(X_batch)

            recon_loss = mse_recon(x_recon, X_target)
            reg_loss   = mse_reg(teff_pred.squeeze(), teff_batch)

            loss = recon_loss + reg_loss

            val_loss += loss.item() * X_batch.size(0)

    val_loss /= len(val_loader.dataset)

    print(f"Epoch [{epoch+1}/{num_epochs}]  "
          f"Train Loss: {train_loss:.6f}  |  Val Loss: {val_loss:.6f}")

    # Early Stopping
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        epochs_no_improve = 0
        torch.save(model.state_dict(), "best_autoencoder_reg.pth")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stop !")
            break


NameError: name 'torch' is not defined

In [7]:
model.load_state_dict(torch.load("best_autoencoder_reg.pth"))
model.eval()

with torch.no_grad():
    # On encode toutes les données (features) pour avoir le latent
    full_data_gpu = torch.tensor(data.values, dtype=torch.float32).to(device)
    latent_2d = model.encoder(full_data_gpu).cpu().numpy()

print("Latent space shape :", latent_2d.shape)  # (N, 2)
print("Embedding sample:\n", latent_2d[:5])


Latent space shape : (175809, 2)
Embedding sample:
 [[-0.4211489   0.31596196]
 [-0.23150465 -0.23192687]
 [ 1.0207897  -1.2943134 ]
 [ 0.48942435 -0.72313267]
 [ 0.3421574  -0.6332825 ]]


In [8]:
with torch.no_grad():
    x_recon, teff_pred = model(full_data_gpu)
    teff_pred = teff_pred.cpu().numpy().flatten()

print("Predicted TEFF shape :", teff_pred.shape)
print("Predicted TEFF samples :", teff_pred[:5])


Predicted TEFF shape : (175809,)
Predicted TEFF samples : [ 0.6461835   0.18968704 -1.4104656  -0.5941096  -0.3766239 ]


In [ ]:
plt.figure(figsize=(10,8), dpi=150)
plt.scatter(latent_2d[:,0], latent_2d[:,1], c=teff, cmap='viridis', s=0.5)
plt.colorbar(label='TEFF (original)')
plt.xlabel("AE1")
plt.ylabel("AE2")
plt.title("Autoencoder 2D + TEFF regression")
plt.show()
